# Maintainability rating — shots: lora

Rates the Code4ML answers of `lora-1shot` and `lora-3shot` (seeds 42, 43, 44) with SonarQube + MLScent, the same
calls as `experiment/experiment/rating/rate_outputs.ipynb`, and uploads `<run>/code4ml_ratings.csv` next to each run on
the Hub.

- Only rows with `ast_valid = True` are rated; the other rows stay in the CSV with empty ratings.
- No MLScent notebook context (model answers do not come from a Kaggle notebook).
- Runs not generated yet or already rated are skipped, so rerunning never rates a run twice.

**Before running:** start both SonarQube API containers (`sonar_api1`, `sonar_api2`) and both MLScent containers
(`mlscent-1`, `mlscent-2`), and log in to Hugging Face with write access.

In [3]:
import csv
import json
import sys
import threading
from collections import Counter
from pathlib import Path

import httpx
from huggingface_hub import HfApi, hf_hub_download

HERE = Path.cwd().resolve()                                   # experiment/experiment/shot-testing/<group>
RATING_PIPELINE = HERE.parents[2] / 'rating' / 'pipeline'     # experiment/rating/pipeline: rate_annotated.py
assert (RATING_PIPELINE / 'rate_annotated.py').exists(), f'rate_annotated.py not found in {RATING_PIPELINE}'
sys.path.insert(0, str(RATING_PIPELINE))
from rate_annotated import merge, mlscent_sender, rate_half, sonar_sender, split_halves  # noqa: E402

OUTPUT_REPO = 'dimsmauls/code4ml-eval-outputs'
RUNS = [f'lora-{k}shot/seed-{s}' for k in (1, 3) for s in (42, 43, 44)]
SONAR = ['http://localhost:8000', 'http://localhost:8001']
MLSCENT = ['http://localhost:8101', 'http://localhost:8102']
BATCH_SIZE = 50
WORK_DIR = HERE / 'results'
RATINGS_FILE = 'code4ml_ratings.csv'

In [4]:
api = HfApi()
on_hub = set(api.list_repo_files(OUTPUT_REPO, repo_type='dataset'))
not_generated = [run for run in RUNS if f'{run}/code4ml_outputs.jsonl' not in on_hub]
todo = [run for run in RUNS if run not in not_generated and f'{run}/{RATINGS_FILE}' not in on_hub]
print('not generated yet (skipped):', not_generated or 'none')
print('to rate:', todo or 'none')
if todo:
    for url in SONAR + MLSCENT:
        httpx.get(f'{url}/health', timeout=10).raise_for_status()
    print('SonarQube and MLScent are up')

not generated yet (skipped): none
to rate: ['lora-1shot/seed-42', 'lora-1shot/seed-43', 'lora-1shot/seed-44', 'lora-3shot/seed-42', 'lora-3shot/seed-43', 'lora-3shot/seed-44']
SonarQube and MLScent are up


In [5]:
def to_csv(run):
    '''Download <run>/code4ml_outputs.jsonl and write it as the CSV rate_annotated expects (id, code_block).'''
    path = hf_hub_download(OUTPUT_REPO, f'{run}/code4ml_outputs.jsonl', repo_type='dataset')
    with open(path) as handle:
        rows = [json.loads(line) for line in handle]
    out = WORK_DIR / run.replace('/', '_') / RATINGS_FILE
    out.parent.mkdir(parents=True, exist_ok=True)
    fields = ['id', 'graph_vertex_class', 'graph_vertex_subclass', 'ast_valid', 'code_block']
    with out.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({'id': row['id'], 'graph_vertex_class': row['graph_vertex_class'],
                             'graph_vertex_subclass': row['graph_vertex_subclass'],
                             'ast_valid': row['ast_valid'], 'code_block': row['code']})
    return out


def rate(csv_path):
    '''Rate the AST-valid rows on both instance pairs in parallel, then merge the ratings into csv_path.'''
    with csv_path.open(newline='', encoding='utf-8') as handle:
        rows = [row for row in csv.DictReader(handle) if row['ast_valid'] == 'True']
    parts = [csv_path.parent / 'part_1.csv', csv_path.parent / 'part_2.csv']
    errors = []

    def work(half, sonar_url, mlscent_url, out):
        try:  # no kernel_of / imports: MLScent gets no notebook context
            rate_half(half, {}, out, sonar_sender(sonar_url), mlscent_sender(mlscent_url), BATCH_SIZE)
        except BaseException as exc:  # noqa: BLE001 - reported after both threads stop
            errors.append(exc)

    threads = [threading.Thread(target=work, args=args)
               for args in zip(split_halves(rows), SONAR, MLSCENT, parts)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise RuntimeError(f'rating failed (rerun to resume): {errors}')
    merge(csv_path, parts, require_all=False)  # AST-invalid rows keep empty ratings

In [6]:
for run in todo:
    csv_path = to_csv(run)
    rate(csv_path)
    api.upload_file(path_or_fileobj=str(csv_path), path_in_repo=f'{run}/{RATINGS_FILE}', repo_id=OUTPUT_REPO,
                    repo_type='dataset', commit_message=f'rating {run}')
    with csv_path.open(newline='', encoding='utf-8') as handle:
        rated = [row for row in csv.DictReader(handle) if row['ast_valid'] == 'True']
    grades = dict(sorted(Counter(row['combined_rating'] for row in rated).items()))
    print(f'{run}: {len(rated)} rated | combined_rating {grades}')
print(f'done: https://huggingface.co/datasets/{OUTPUT_REPO}')

part_1.csv: 50/152
part_2.csv: 50/152
part_1.csv: 100/152
part_2.csv: 100/152
part_1.csv: 150/152
part_2.csv: 150/152
part_2.csv: 152/152
part_1.csv: 152/152
lora-1shot/seed-42: 304 rated | combined_rating {'A': 14, 'B': 34, 'C': 29, 'D': 1, 'N/A': 226}
part_2.csv: 50/150
part_1.csv: 50/151
part_2.csv: 100/150
part_1.csv: 100/151
part_2.csv: 150/150
part_1.csv: 150/151
part_1.csv: 151/151
lora-1shot/seed-43: 301 rated | combined_rating {'A': 11, 'B': 61, 'C': 1, 'N/A': 228}
part_1.csv: 50/152
part_2.csv: 50/152
part_1.csv: 100/152
part_2.csv: 100/152
part_1.csv: 150/152
part_2.csv: 150/152
part_1.csv: 152/152
part_2.csv: 152/152
lora-1shot/seed-44: 304 rated | combined_rating {'A': 11, 'B': 21, 'C': 41, 'D': 1, 'N/A': 230}
part_2.csv: 50/152
part_1.csv: 50/152
part_1.csv: 100/152
part_2.csv: 100/152
part_1.csv: 150/152
part_2.csv: 150/152
part_2.csv: 152/152
part_1.csv: 152/152
lora-3shot/seed-42: 304 rated | combined_rating {'A': 11, 'B': 47, 'C': 16, 'N/A': 230}
part_1.csv: 50/149
pa